# MCQ Generator using Groq API

This notebook demonstrates the MCQ generation process for three categories:
- Agriculture
- Climate
- Renewable Energy

Each category will generate 3 sets of MCQs with 5 questions each.

## 1. Import Required Libraries

In [2]:

import os
import json
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq
import pypdf
from typing import List, Dict, Any
import random

## 2. Load Environment Variables

In [3]:

# Load environment variables from .env file
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in environment variables")



## 3. Initialize Groq Client

In [3]:

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)
print("✓ Groq client initialized")

✓ Groq client initialized


## 4. Define PDF Parsing Function

In [4]:

def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract text content from a PDF file.
    
    Args:
        pdf_path: Path to the PDF file
        
    Returns:
        Extracted text content from the PDF
        
    Security: Validates file path to prevent directory traversal attacks.
    """
    try:
        # Validate file exists and is a PDF
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"PDF file not found: {pdf_path}")
        
        if not pdf_path.lower().endswith('.pdf'):
            raise ValueError("File must be a PDF")
        
        text_content = ""
        with open(pdf_path, 'rb') as file:
            pdf_reader = pypdf.PdfReader(file)
            num_pages = len(pdf_reader.pages)
            print(f"  Processing {pdf_path.split(os.sep)[-1]} - {num_pages} pages")
            
            for page_num in range(num_pages):
                page = pdf_reader.pages[page_num]
                text_content += page.extract_text() + "\n"
        
        return text_content.strip()
    except Exception as e:
        print(f"Error extracting text from {pdf_path}: {str(e)}")
        return ""

## 5. Test PDF Extraction

In [5]:

# Test PDF extraction with one file
base_path = Path(r"C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF\data")
test_pdf = base_path / "agriculture" / "agri_1.pdf"

if test_pdf.exists():
    sample_text = extract_text_from_pdf(str(test_pdf))
    print(f"\nExtracted {len(sample_text)} characters")
    print(f"\nFirst 500 characters:\n{sample_text[:500]}")
else:
    print(f"Test file not found: {test_pdf}")

  Processing agri_1.pdf - 6 pages

Extracted 31009 characters

First 500 characters:
© 2024 IJSRET 
2511 
 
International Journal of Scientific Research & Engineering Trends                                                                                                         
Volume 10, Issue 5, Sept-Oct-2024, ISSN (Online): 2395-566X 
 
Agriculture Sustainability: A Comprehensive Review  
Rajat Kumar, Gurshaminder Singh 
University Institute of Agricultural Sciences,  
Chandigarh University, Gharuan, Punjab, India Kumar, Gurshaminder Singh 
 
Abstract- Agricultural sustainabi


## 6. Define MCQ Generation Function

In [9]:

def generate_mcq_set(client: Groq, content: str, category: str, set_number: int, num_questions: int = 5) -> Dict[str, Any]:
    """
    Generate a set of MCQ questions using Groq API.
    
    Args:
        client: Initialized Groq client
        content: Text content from which to generate questions
        category: Category name (agriculture, climate, renewable_energy)
        set_number: Set number (1, 2, or 3)
        num_questions: Number of questions per set (default: 5)
        
    Returns:
        Dictionary containing MCQ set information
        
    Security: Sanitizes user input to prevent prompt injection attacks.
    Resource Efficiency: Uses efficient token usage with truncated content.
    """
    # Truncate content to manage token limits (approximately 8000 tokens worth of text)
    max_chars = 25000
    if len(content) > max_chars:
        # Take content from different sections for variety
        segment_size = max_chars // 3
        content = (
            content[:segment_size] + 
            content[len(content)//2 - segment_size//2:len(content)//2 + segment_size//2] + 
            content[-segment_size:]
        )
    
    # Sanitize category name to prevent injection
    category_clean = category.replace('_', ' ').title()
    
    prompt = f"""Based on the following content about {category_clean}, create {num_questions} multiple-choice questions.

Content:
{content}

Generate exactly {num_questions} multiple-choice questions. Each question must have:
1. A clear question text
2. Four options (A, B, C, D)
3. The correct answer (letter only: A, B, C, or D)
4. A brief explanation of why the answer is correct

Format your response ONLY as a valid JSON array. Do not include any markdown formatting or code blocks. Just return the raw JSON array.

Example format:
[
  {{
    "question": "What is...",
    "options": {{
      "A": "Option 1",
      "B": "Option 2",
      "C": "Option 3",
      "D": "Option 4"
    }},
    "correct_answer": "B",
    "explanation": "The correct answer is B because..."
  }}
]
"""
    
    try:
        # Call Groq API
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are an expert educator creating high-quality multiple-choice questions. Always respond with valid JSON only, no markdown or code blocks."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            model="llama-3.3-70b-versatile",
            temperature=0.7,
            max_tokens=2048
        )
        
        response_text = chat_completion.choices[0].message.content.strip()
        
        # Remove markdown code blocks if present
        if response_text.startswith("```"):
            lines = response_text.split("\n")
            response_text = "\n".join(lines[1:-1]) if len(lines) > 2 else response_text
            response_text = response_text.replace("```json", "").replace("```", "").strip()
        
        # Parse JSON response
        questions = json.loads(response_text)
        
        return {
            "category": category,
            "set_number": set_number,
            "questions": questions,
            "total_questions": len(questions)
        }
        
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON response: {e}")
        print(f"Response was: {response_text[:500]}")
        return None
    except Exception as e:
        print(f"Error generating MCQ set: {e}")
        return None

## 7. Define Category Processing Function

In [10]:

def process_category(client: Groq, category: str, data_path: Path) -> List[Dict[str, Any]]:
    """
    Process all PDFs in a category and generate 3 sets of MCQs.
    
    Args:
        client: Initialized Groq client
        category: Category name (agriculture, climate, renewable_energy)
        data_path: Base path to data directory
        
    Returns:
        List of 3 MCQ sets, each containing 5 questions
        
    Security: Validates paths to prevent directory traversal.
    Resource Efficiency: Processes PDFs once and reuses content.
    """
    print(f"\n{'='*60}")
    print(f"Processing Category: {category.upper()}")
    print(f"{'='*60}")
    
    # Validate category name to prevent path traversal
    allowed_categories = ['agriculture', 'climate', 'renewable_energy']
    if category not in allowed_categories:
        raise ValueError(f"Invalid category. Must be one of: {allowed_categories}")
    
    category_path = data_path / category
    
    if not category_path.exists():
        print(f"Category path not found: {category_path}")
        return []
    
    # Get all PDF files in the category
    pdf_files = list(category_path.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files")
    
    if not pdf_files:
        print(f"No PDF files found in {category_path}")
        return []
    
    # Extract text from all PDFs
    combined_content = ""
    for pdf_file in pdf_files:
        text = extract_text_from_pdf(str(pdf_file))
        combined_content += text + "\n\n"
    
    print(f"\nTotal extracted content: {len(combined_content)} characters")
    
    # Generate 3 sets of MCQs
    mcq_sets = []
    for set_num in range(1, 4):
        print(f"\nGenerating MCQ Set {set_num}...")
        mcq_set = generate_mcq_set(client, combined_content, category, set_num)
        
        if mcq_set:
            mcq_sets.append(mcq_set)
            print(f"✓ Set {set_num} generated successfully with {mcq_set['total_questions']} questions")
        else:
            print(f"✗ Failed to generate Set {set_num}")
    
    return mcq_sets

## 8. Test MCQ Generation for Single Category

In [11]:

# Test with Agriculture category
data_path = Path(r"C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF\data")

agriculture_mcqs = process_category(client, "agriculture", data_path)


Processing Category: AGRICULTURE
Found 2 PDF files
  Processing agri_1.pdf - 6 pages
  Processing agri_2.pdf - 79 pages

Total extracted content: 214294 characters

Generating MCQ Set 1...
✓ Set 1 generated successfully with 5 questions

Generating MCQ Set 2...
✓ Set 2 generated successfully with 5 questions

Generating MCQ Set 3...
✓ Set 3 generated successfully with 5 questions


## 9. Display Generated MCQs

In [12]:

# Display the first set nicely formatted
if agriculture_mcqs:
    first_set = agriculture_mcqs[0]
    print(f"\nCategory: {first_set['category']}")
    print(f"Set Number: {first_set['set_number']}")
    print(f"\n{'='*60}\n")
    
    for idx, q in enumerate(first_set['questions'], 1):
        print(f"Question {idx}: {q['question']}")
        for opt_key, opt_val in q['options'].items():
            marker = "✓" if opt_key == q['correct_answer'] else " "
            print(f"  {marker} {opt_key}. {opt_val}")
        print(f"  Correct Answer: {q['correct_answer']}")
        print(f"  Explanation: {q['explanation']}")
        print(f"\n{'-'*60}\n")


Category: agriculture
Set Number: 1


Question 1: What is the primary goal of agricultural sustainability?
    A. To increase crop yields at any cost
  ✓ B. To balance productivity with environmental preservation and social fairness
    C. To reduce the use of synthetic fertilizers only
    D. To promote the use of genetically modified organisms (GMOs)
  Correct Answer: B
  Explanation: The correct answer is B because agricultural sustainability aims to meet current food needs without compromising the ability of future generations to meet their own needs, while also considering ecological integrity, economic viability, and social equity.

------------------------------------------------------------

Question 2: What is the main benefit of agroecology in sustainable agriculture?
    A. Increased use of chemical pesticides
    B. Improved crop yields through monocultures
  ✓ C. Enhanced biodiversity and ecosystem services
    D. Reduced water retention in the soil
  Correct Answer: C
  

## 10. Generate MCQs for All Categories

In [13]:

# Process all three categories
categories = ['agriculture', 'climate', 'renewable_energy']
all_mcqs = {}

for category in categories:
    mcq_sets = process_category(client, category, data_path)
    all_mcqs[category] = mcq_sets

print(f"\n{'='*60}")
print("GENERATION COMPLETE")
print(f"{'='*60}")
for cat, sets in all_mcqs.items():
    print(f"{cat.upper()}: {len(sets)} sets generated")


Processing Category: AGRICULTURE
Found 2 PDF files
  Processing agri_1.pdf - 6 pages
  Processing agri_2.pdf - 79 pages

Total extracted content: 214294 characters

Generating MCQ Set 1...
✓ Set 1 generated successfully with 5 questions

Generating MCQ Set 2...
✓ Set 2 generated successfully with 5 questions

Generating MCQ Set 3...
✓ Set 3 generated successfully with 5 questions

Processing Category: CLIMATE
Found 2 PDF files
  Processing climate_change_1.pdf - 12 pages
  Processing climate_change_2.pdf - 20 pages

Total extracted content: 53176 characters

Generating MCQ Set 1...
✓ Set 1 generated successfully with 5 questions

Generating MCQ Set 2...
✓ Set 2 generated successfully with 5 questions

Generating MCQ Set 3...
✓ Set 3 generated successfully with 5 questions

Processing Category: RENEWABLE_ENERGY
Found 2 PDF files
  Processing renewable_energy_1.pdf - 36 pages
  Processing renewable_energy_2.pdf - 35 pages

Total extracted content: 208943 characters

Generating MCQ Set 1

## 11. Save Results to JSON File

In [14]:

# Save all MCQs to a JSON file
output_file = Path(r"C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF\data\mcq_output") / "mcq_results.json"

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_mcqs, f, indent=2, ensure_ascii=False)

print(f"✓ Results saved to: {output_file}")
print(f"File size: {output_file.stat().st_size / 1024:.2f} KB")

✓ Results saved to: C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF\data\mcq_output\mcq_results.json
File size: 31.13 KB


## 12. Test Category-Specific Retrieval (API Simulation)

In [ ]:

# Simulate API request for a specific category
def get_mcqs_for_category(category: str, all_mcqs: Dict[str, List]) -> Dict[str, Any]:
    """
    Retrieve MCQs for a specific category (simulates API response).
    
    Args:
        category: Category name
        all_mcqs: Dictionary containing all generated MCQs
        
    Returns:
        Dictionary with category MCQs or error message
        
    Security: Validates category input against whitelist.
    """
    # Input validation
    allowed_categories = ['agriculture', 'climate', 'renewable_energy']
    if category not in allowed_categories:
        return {
            "status": "error",
            "message": f"Invalid category. Must be one of: {allowed_categories}"
        }
    
    if category not in all_mcqs or not all_mcqs[category]:
        return {
            "status": "error",
            "message": f"No MCQs found for category: {category}"
        }
    
    return {
        "status": "success",
        "category": category,
        "mcq_sets": all_mcqs[category],
        "total_sets": len(all_mcqs[category])
    }

# Test retrieval
test_category = "climate"
result = get_mcqs_for_category(test_category, all_mcqs)
print(json.dumps(result, indent=2))

## 13. Validate MCQ Structure

In [15]:

def validate_mcq_structure(mcq_sets: List[Dict]) -> bool:
    """
    Validate that MCQ sets follow the expected structure.
    
    Args:
        mcq_sets: List of MCQ sets to validate
        
    Returns:
        True if valid, False otherwise
    """
    required_keys = ['category', 'set_number', 'questions', 'total_questions']
    question_keys = ['question', 'options', 'correct_answer', 'explanation']
    
    for mcq_set in mcq_sets:
        # Validate set structure
        if not all(key in mcq_set for key in required_keys):
            print(f"✗ Missing required keys in set {mcq_set.get('set_number', '?')}")
            return False
        
        # Validate questions
        for q_idx, question in enumerate(mcq_set['questions'], 1):
            if not all(key in question for key in question_keys):
                print(f"✗ Missing required keys in question {q_idx}")
                return False
            
            # Validate options
            if not all(opt in question['options'] for opt in ['A', 'B', 'C', 'D']):
                print(f"✗ Missing option keys in question {q_idx}")
                return False
            
            # Validate correct answer
            if question['correct_answer'] not in ['A', 'B', 'C', 'D']:
                print(f"✗ Invalid correct_answer in question {q_idx}")
                return False
    
    print("✓ All MCQ sets are valid")
    return True

# Validate all generated MCQs
for category, sets in all_mcqs.items():
    print(f"\nValidating {category}...")
    validate_mcq_structure(sets)


Validating agriculture...
✓ All MCQ sets are valid

Validating climate...
✓ All MCQ sets are valid

Validating renewable_energy...
✓ All MCQ sets are valid


## 14. Summary Statistics

In [16]:

# Display summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

total_sets = 0
total_questions = 0

for category, sets in all_mcqs.items():
    cat_questions = sum(s['total_questions'] for s in sets)
    total_sets += len(sets)
    total_questions += cat_questions
    print(f"\n{category.upper()}:")
    print(f"  - Sets: {len(sets)}")
    print(f"  - Total Questions: {cat_questions}")

print(f"\n{'='*60}")
print(f"TOTAL SETS: {total_sets}")
print(f"TOTAL QUESTIONS: {total_questions}")
print(f"{'='*60}")


SUMMARY STATISTICS

AGRICULTURE:
  - Sets: 3
  - Total Questions: 15

CLIMATE:
  - Sets: 3
  - Total Questions: 15

RENEWABLE_ENERGY:
  - Sets: 3
  - Total Questions: 15

TOTAL SETS: 9
TOTAL QUESTIONS: 45
